# 5_TARGETS — Predict Heat-Flow Target Grids

Loads trained models from `4_MODEL` and predicts on Antarctica and  
Greenland (and any future grids added to `TARGET_GRIDS`).  
Outputs one NetCDF per method per grid.

**Sections**
1. Configuration & paths  
2. Helper functions (NetCDF writer, histogram builder)  
3. Similarity prediction + outputs  
4. QRF prediction + outputs  
5. GBM prediction + outputs  
6. Output manifest  


## 1 · Configuration & paths

In [ ]:
import sys, json, pickle, warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.interpolate import PchipInterpolator
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')

from config import *

# ── user-adjustable ─────────────────────────────────────────────────────
local_data   = Path('data')
model_dir    = Path('output/models')
param_dir    = Path('output/sweeps')
out_dir      = Path('output/targets')

# Grids to predict — add more entries later
# Each entry: (grid_key, parquet_path, crs_epsg, x_coord, y_coord)
TARGET_GRIDS = [
    ('ant', local_data / 'antarctica.parquet', 3031, 'x', 'y'),
    ('grl', local_data / 'greenland.parquet',  3413, 'x', 'y'),
]


out_dir.mkdir(parents=True, exist_ok=True)

# ── load parameter JSONs ─────────────────────────────────────────────────
with open(param_dir / 'qrf_params.json') as f:     QP = json.load(f)
with open(param_dir / 'sim_best_params.json') as f: SP = json.load(f)
with open(param_dir / 'gbm_params.json') as f:     GP = json.load(f)



# ── histogram bin edges ──────────────────────────────────────────────────
q_edges = np.arange(Q_CLIP_MIN, q_clip_max + HIST_BIN_W, HIST_BIN_W)
n_bins  = len(q_edges) - 1
assert n_bins <= HIST_MAX_BINS, (
    f'n_bins={n_bins} exceeds HIST_MAX_BINS={HIST_MAX_BINS}. '
    f'Increase HIST_BIN_W or HIST_MAX_BINS.')
q_centers = 0.5 * (q_edges[:-1] + q_edges[1:])
print(f'Features : {len(obs_model)}')
print(f'Hist bins: {n_bins}  ({q_edges[0]:.3f}–{q_edges[-1]:.3f} W/m²)')

Features : 22
Hist bins: 35  (0.000–0.350 W/m²)


## 2 · Helper functions

In [12]:
# ── load reference + fitted artefacts ────────────────────────────────────
print('Loading model artefacts...')
with open(model_dir / 'qrf_model.pkl', 'rb') as f:
    qrf = pickle.load(f)
with open(model_dir / 'sim_correction_spline.pkl', 'rb') as f:
    sim_artefacts = pickle.load(f)   # keys: pred_pctls, true_pctls, scaler, sigma_arr, K

gbm_models = {}
for q in GP['quantile_losses']:
    with open(model_dir / f'gbm_q{int(q*100):02d}_model.pkl', 'rb') as f:
        gbm_models[q] = pickle.load(f)

scaler    = sim_artefacts['scaler']
sigma_arr = sim_artefacts['sigma_arr']
K         = sim_artefacts['K']
_pp = sim_artefacts['pred_pctls']
_tp = sim_artefacts['true_pctls']
_, _keep = np.unique(_pp, return_index=True)
correction_spline = PchipInterpolator(_pp[_keep], _tp[_keep], extrapolate=True)

# ── load reference parquet for similarity ────────────────────────────────
parquet_ref = local_data / 'IHFC_obs.parquet'
df_ref_full = pd.read_parquet(parquet_ref)
df_ref = df_ref_full.dropna(subset=obs_model + ['q']).query(f'q > {Q_CLIP_MIN} & q <= {q_clip_max}')
X_ref  = scaler.transform(df_ref[obs_model].values.astype(np.float32)) / sigma_arr
y_ref  = df_ref['q'].values.astype(np.float32)

# Spatial density weights — same column written by 1_Import
if 'weight' in df_ref.columns:
    w_ref = df_ref['weight'].values.astype(np.float32)
    w_ref = w_ref / w_ref.mean()
    print(f'Reference weights loaded  min={w_ref.min():.4f}  max={w_ref.max():.4f}')
else:
    w_ref = np.ones(len(y_ref), dtype=np.float32)
    print('WARNING: no weight column — using uniform reference weights')

print(f'Reference samples: {len(y_ref)}')

# ── conformal quantiles loaded from 4_MODEL run ──────────────────────────
# (Stored as part of sim_artefacts dict if available, else recompute)
qhat_qrf = sim_artefacts.get('qhat_qrf', None)
qhat_gbm = sim_artefacts.get('qhat_gbm', None)
if qhat_qrf is None:
    print('⚠  Conformal qhats not found in artefacts — run 4_MODEL first.')
    print('   PI outputs will use raw Q05/Q95 without conformal adjustment.')


def make_ds_2d(df_grid, data_vars: dict, attrs: dict, crs_epsg: int,
               x_col='x', y_col='y') -> xr.Dataset:
    """Build a 2-D xarray Dataset from a flat DataFrame + data_vars dict.

    Assumes the parquet was created on a regular grid so that unique
    (x, y) values reshape cleanly into a 2-D array.
    Variables in data_vars that have shape (n, n_bins) become 3-D (y, x, bin).
    """
    xs = np.sort(df_grid[x_col].unique())
    ys = np.sort(df_grid[y_col].unique())
    nx, ny = len(xs), len(ys)
    xi = np.searchsorted(xs, df_grid[x_col].values)
    yi = np.searchsorted(ys, df_grid[y_col].values)

    coords = {x_col: xs, y_col: ys}
    dvs    = {}
    for name, arr in data_vars.items():
        if arr.ndim == 1:
            grid = np.full((ny, nx), np.nan, dtype=np.float32)
            grid[yi, xi] = arr
            dvs[name] = xr.DataArray(grid, dims=[y_col, x_col])
        elif arr.ndim == 2:   # (n_pts, n_bins) → (y, x, bin)
            grid = np.full((ny, nx, arr.shape[1]), np.nan, dtype=np.float32)
            grid[yi, xi, :] = arr
            dvs[name] = xr.DataArray(grid, dims=[y_col, x_col, 'bin'],
                                      coords={'bin': q_centers})
    ds = xr.Dataset(dvs, coords=coords, attrs=attrs)
    ds.attrs['crs'] = f'EPSG:{crs_epsg}'
    ds.attrs['q_clip_min'] = float(Q_CLIP_MIN)
    ds.attrs['q_clip_max'] = float(q_clip_max)
    return ds


def hist_from_samples(samples_2d: np.ndarray) -> np.ndarray:
    """Convert (n_pts, n_trees) array of samples to (n_pts, n_bins) histograms."""
    hists = np.zeros((len(samples_2d), n_bins), dtype=np.float32)
    for i, row in enumerate(samples_2d):
        h, _ = np.histogram(row, bins=q_edges)
        s = h.sum()
        hists[i] = h / s if s > 0 else h
    return hists

print('Helpers ready.')

Loading model artefacts...
Reference weights loaded  min=0.1123  max=3.5262
Reference samples: 30847
⚠  Conformal qhats not found in artefacts — run 4_MODEL first.
   PI outputs will use raw Q05/Q95 without conformal adjustment.
Helpers ready.


## 3 · Similarity prediction

Outputs per grid:
- `Q_mean`   — kernel-weighted mean (mean-reversion corrected)
- `Q_median` — kernel-weighted median (corrected)
- `Q_std`    — kernel-weighted standard deviation
- `N_eff`    — effective number of reference samples (1/Σw²)
- `Q_hist`   — 3-D normalised histogram (y, x, bin)

In [ ]:
def sim_predict_grid(X_scaled):
    """Batched Similarity kernel; returns (n, 4+n_bins) array."""
    means, meds, stds, neffs, hists = [], [], [], [], []
    sort_idx = np.argsort(y_ref)
    y_s      = y_ref[sort_idx]
    for i in range(0, len(X_scaled), BATCH_SIM):
        Xb   = X_scaled[i:i+BATCH_SIM]
        diff = (Xb[:, None, :] - X_ref[None, :, :]) / sigma_arr
        S    = np.exp(-0.5 * np.mean(diff**2, axis=2))
        S_K  = S ** K
        # multiply by spatial density weight before normalising
        w    = S_K * w_ref[None, :]                                # apply ref weights
        w    = w / (w.sum(axis=1, keepdims=True) + 1e-30)          # normalise
        q_mean = (w * y_ref).sum(axis=1)
        q_std  = np.sqrt((w * (y_ref - q_mean[:,None])**2).sum(axis=1))
        w_s    = w[:, sort_idx]
        cumw   = w_s.cumsum(axis=1)
        med_idx = np.argmax(cumw >= 0.5, axis=1)
        q_med  = y_s[med_idx]
        n_eff  = 1.0 / (w**2).sum(axis=1)
        # per-point weighted histograms
        h_batch = np.zeros((len(Xb), n_bins), dtype=np.float32)
        for j in range(len(Xb)):
            bin_idx = np.searchsorted(q_edges[1:], y_ref)
            bin_idx = np.clip(bin_idx, 0, n_bins-1)
            np.add.at(h_batch[j], bin_idx, w[j])
        h_batch /= (h_batch.sum(axis=1, keepdims=True) + 1e-30)
        means.append(q_mean); meds.append(q_med)
        stds.append(q_std);   neffs.append(n_eff)
        hists.append(h_batch)
    return (np.concatenate(means), np.concatenate(meds),
            np.concatenate(stds),  np.concatenate(neffs),
            np.vstack(hists))


for grid_key, parquet_path, epsg, xc, yc in TARGET_GRIDS:
    print(f'\n── Similarity  [{grid_key}] ─────────────────────────')
    df_g  = pd.read_parquet(parquet_path).dropna(subset=obs_model)
    X_g   = scaler.transform(df_g[obs_model].values.astype(np.float32)) / sigma_arr

    q_mean, q_med, q_std, n_eff, q_hist = sim_predict_grid(X_g)
    q_mean_c = correction_spline(q_mean).astype(np.float32)
    q_med_c  = correction_spline(q_med).astype(np.float32)

    ds = make_ds_2d(df_g, {
        'Q_mean'  : q_mean_c,
        'Q_median': q_med_c,
        'Q_std'   : q_std.astype(np.float32),
        'N_eff'   : n_eff.astype(np.float32),
        'Q_hist'  : q_hist,
    }, attrs={
        'method'     : 'Similarity kernel (mean-reversion corrected)',
        'K'          : float(K),
        'source'     : str(parquet_path),
    }, crs_epsg=epsg, x_col=xc, y_col=yc)

    nc_path = out_dir / f'sim_{grid_key}.nc'
    ds.to_netcdf(nc_path)
    print(f'  saved → {nc_path}')
    print(f'  Q_mean range: {float(q_mean_c.min())*1e3:.1f}–{float(q_mean_c.max())*1e3:.1f} mW/m²')
print('Similarity done.')


── Similarity  [ant] ─────────────────────────


## 4 · QRF prediction

Outputs per grid:
- `Q_q05`, `Q_q25`, `Q_q50`, `Q_q75`, `Q_q95` — quantile estimates
- `Q_mean`, `Q_std`, `Q_skew` — moments of the empirical leaf distribution
- `PI90_width` — raw Q95−Q05
- `PI90_lo`, `PI90_hi` — conformal-adjusted bounds (if qhat available)
- `Q_hist` — 3-D empirical distribution histogram (y, x, bin)

In [7]:
from scipy.stats import skew as scipy_skew

for grid_key, parquet_path, epsg, xc, yc in TARGET_GRIDS:
    print(f'\n── QRF  [{grid_key}] ────────────────────────────────')
    df_g = pd.read_parquet(parquet_path).dropna(subset=obs_model)
    X_g  = scaler.transform(df_g[obs_model].values.astype(np.float32))

    # ── quantile predictions ─────────────────────────────────────────────
    qrf_q = qrf.predict(X_g, quantiles=QUANTILES)                       # (n, 5)
    # Dense quantile grid used for mean, std, skew, and histogram
    QUANTILES_DENSE = np.linspace(0.02, 0.98, 49).tolist()
    print('  Sampling dense quantile grid...')
    leaf_samples = qrf.predict(X_g, quantiles=QUANTILES_DENSE)           # (n, 49)
    qrf_m = leaf_samples.mean(axis=1)
    qrf_s = leaf_samples.std(axis=1)
    # leaf_samples: (n, 49)  — good proxy for the empirical distribution
    q_hist_qrf = hist_from_samples(leaf_samples)
    q_skew     = np.array([scipy_skew(row) for row in leaf_samples], dtype=np.float32)

    # ── conformal PI ─────────────────────────────────────────────────────
    q05 = qrf_q[:, 0]; q95 = qrf_q[:, -1]
    if qhat_qrf is not None:
        pi_lo = (q05 - qhat_qrf).astype(np.float32)
        pi_hi = (q95 + qhat_qrf).astype(np.float32)
    else:
        pi_lo = q05.astype(np.float32)
        pi_hi = q95.astype(np.float32)

    ds = make_ds_2d(df_g, {
        'Q_q05'     : qrf_q[:,0].astype(np.float32),
        'Q_q25'     : qrf_q[:,1].astype(np.float32),
        'Q_q50'     : qrf_q[:,2].astype(np.float32),
        'Q_q75'     : qrf_q[:,3].astype(np.float32),
        'Q_q95'     : qrf_q[:,4].astype(np.float32),
        'Q_mean'    : qrf_m.astype(np.float32),
        'Q_std'     : qrf_s.astype(np.float32),
        'Q_skew'    : q_skew,
        'PI90_width': (q95 - q05).astype(np.float32),
        'PI90_lo'   : pi_lo,
        'PI90_hi'   : pi_hi,
        'Q_hist'    : q_hist_qrf,
    }, attrs={
        'method'   : 'Quantile Random Forest',
        'source'   : str(parquet_path),
        'quantiles': str(QUANTILES),
        'conformal_alpha': CONFORMAL_ALPHA,
    }, crs_epsg=epsg, x_col=xc, y_col=yc)

    nc_path = out_dir / f'qrf_{grid_key}.nc'
    ds.to_netcdf(nc_path)
    print(f'  saved → {nc_path}')
    print(f'  Q_q50 range: {float(qrf_q[:,2].min())*1e3:.1f}–{float(qrf_q[:,2].max())*1e3:.1f} mW/m²')
print('QRF done.')


── QRF  [ant] ────────────────────────────────
  Sampling dense quantile grid...
  saved → output/targets/qrf_ant.nc
  Q_q50 range: 32.0–114.0 mW/m²

── QRF  [grl] ────────────────────────────────


KeyError: ['VP_40_VS_50', 'VP_120_VS_140']

## 5 · GBM prediction

Outputs per grid:
- `Q_q05`, `Q_q50`, `Q_q95` — quantile model predictions
- `PI90_width` — Q95−Q05
- `PI90_lo`, `PI90_hi` — conformal-adjusted 90% bounds

In [ ]:
for grid_key, parquet_path, epsg, xc, yc in TARGET_GRIDS:
    print(f'\n── GBM  [{grid_key}] ────────────────────────────────')
    df_g = pd.read_parquet(parquet_path).dropna(subset=obs_model)
    X_g  = scaler.transform(df_g[obs_model].values.astype(np.float32))

    g_q05 = gbm_models[0.05].predict(X_g).astype(np.float32)
    g_q50 = gbm_models[0.50].predict(X_g).astype(np.float32)
    g_q95 = gbm_models[0.95].predict(X_g).astype(np.float32)

    if qhat_gbm is not None:
        pi_lo = (g_q05 - qhat_gbm).astype(np.float32)
        pi_hi = (g_q95 + qhat_gbm).astype(np.float32)
    else:
        pi_lo, pi_hi = g_q05, g_q95

    ds = make_ds_2d(df_g, {
        'Q_q05'     : g_q05,
        'Q_q50'     : g_q50,
        'Q_q95'     : g_q95,
        'PI90_width': (g_q95 - g_q05).astype(np.float32),
        'PI90_lo'   : pi_lo,
        'PI90_hi'   : pi_hi,
    }, attrs={
        'method'         : 'HistGradientBoosting (quantile + conformal)',
        'source'         : str(parquet_path),
        'conformal_alpha': CONFORMAL_ALPHA,
    }, crs_epsg=epsg, x_col=xc, y_col=yc)

    nc_path = out_dir / f'gbm_{grid_key}.nc'
    ds.to_netcdf(nc_path)
    print(f'  saved → {nc_path}')
    print(f'  Q_q50 range: {float(g_q50.min())*1e3:.1f}–{float(g_q50.max())*1e3:.1f} mW/m²')
print('GBM done.')

## 6 · Output manifest

In [ ]:
import os
rows = []
for p in sorted(out_dir.glob('*.nc')):
    ds = xr.open_dataset(p)
    rows.append({'file': p.name,
                 'method': ds.attrs.get('method',''),
                 'variables': list(ds.data_vars),
                 'size_MB': round(os.path.getsize(p)/1e6, 2)})
    ds.close()
manifest = pd.DataFrame(rows)
manifest.to_csv(out_dir / 'manifest.csv', index=False)
print(manifest.to_string(index=False))